# FIUBench GA-forget5 평가 — Kaggle T4
최종 checkpoint를 만든 Notebook의 Output을 **Add Input**으로 연결하세요. T4 GPU와 Internet을 켜고 셀을 순서대로 실행합니다. 첫 설치 셀 뒤 커널을 재시작하세요.
평가 범위: 공개 FIUBench 코드의 forget5 APE, forget5 likelihood/EM/Min-K, retain5 ROUGE/Truth Ratio. GPT-Eval, MME, POPE, retain-only baseline KS 통계는 이 노트북에 포함되지 않으므로 논문 전체 점수로 해석하지 마세요.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then git clone https://github.com/npngjjm/FIUBench_kaggle.git; fi
python -m pip uninstall -y torchao || true
python -m pip install 'transformers==4.37.2' 'tokenizers==0.15.1' 'peft==0.8.2' 'accelerate==1.14.0' 'deepspeed==0.19.7' 'huggingface_hub==0.36.2' 'numpy==2.1.3' hydra-core omegaconf datasets sentencepiece protobuf scipy pillow pyyaml requests tqdm rouge_score scikit-learn

커널을 한 번 재시작한 뒤 아래 셀부터 실행하세요. 설치 셀을 다시 실행할 필요는 없습니다.

In [ ]:
import os, shutil, json
from pathlib import Path
from importlib.metadata import version
import torch
repo=Path('/kaggle/working/FIUBench_kaggle')
os.chdir(repo)
os.environ['HF_HOME']='/kaggle/working/hf_cache'
expected={'transformers':'4.37.2','tokenizers':'0.15.1','peft':'0.8.2','accelerate':'1.14.0','deepspeed':'0.19.7','huggingface-hub':'0.36.2','numpy':'2.1.3'}
for package,want in expected.items():
    have=version(package)
    print(package,have)
    assert have==want,(package,have)
print('torch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0))
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
# Add Input에 연결된 최종 checkpoint만 찾습니다. 중간 step_* 결과는 제외.
candidates=[p for p in Path('/kaggle/input').rglob('checkpoint.pt') if not any(x.startswith('step_') for x in p.parts)]
print('최종 checkpoint 후보:',*[str(p) for p in candidates],sep='\n')
assert len(candidates)==1,'최종 checkpoint 후보가 정확히 하나여야 합니다. Add Input 또는 후보 경로를 확인하세요.'
target=Path('/kaggle/working/ga_forget5_checkpoint/checkpoint.pt')
target.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(candidates[0],target)
state=torch.load(target,map_location='cpu',weights_only=True)
assert isinstance(state,dict) and state
print('checkpoint 텐서 수:',len(state),'파일 크기 MiB:',round(target.stat().st_size/1024**2,1))
del state

In [ ]:
# 원본 데이터 JSON을 유지하고 공개 SFHQ archive에서 JSON 경로가 요구하는 이미지를 복원합니다.
import json, shutil, zipfile
from pathlib import Path
from collections import defaultdict
from huggingface_hub import HfApi,hf_hub_download
from PIL import Image
repo=Path('/kaggle/working/FIUBench_kaggle')
records=[json.loads(s) for s in (repo/'dataset/full.json').read_text().splitlines() if s.strip()][:400]
split=json.loads((repo/'dataset/split.json').read_text())
needed={r['image_path'] for r in records if r['unique_id'] in set(split['forget5']+split['retain5'])}
assert len(needed)==40
rev=HfApi().dataset_info('gray311/FIUBench').sha
archive=hf_hub_download('gray311/FIUBench','SFHQ.zip',repo_type='dataset',revision=rev,local_dir='/kaggle/working/fiu_download')
with zipfile.ZipFile(archive) as z:
    entries=defaultdict(list)
    for m in z.infolist():
        p=Path(m.filename)
        if not m.is_dir() and '__MACOSX' not in p.parts and not p.name.startswith('._'):
            entries[p.name].append(m)
    for image_path in needed:
        dest=(repo/image_path).resolve()
        assert dest.is_relative_to((repo/'dataset').resolve())
        hits=entries[dest.name]
        assert len(hits)==1,(dest.name,len(hits))
        dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(hits[0]) as src,dest.open('wb') as dst: shutil.copyfileobj(src,dst)
        with Image.open(dest) as im: im.verify()
assert all((repo/p).is_file() for p in needed)
Path('/kaggle/working/fiu_eval_data_revision.txt').write_text(rev)
print('forget5 + retain5 이미지 검증:',len(needed),'개. dataset revision:',rev)

In [ ]:
# Kaggle 런타임의 evaluate_util.py 복사본에만 필수 호환성 패치를 적용합니다.
import ast
from pathlib import Path
path=Path('/kaggle/working/FIUBench_kaggle/evaluate_util.py')
s=path.read_text()
old='from data_generation.api import (\n    GeminiEvaluator, \n    GPTEvaluator, \n    system_message, \n    user_message, \n    jobs\n)'
shim="""class GPTEvaluator:
    def __init__(self, *args, **kwargs): pass
    def generate_answer(self, *args, **kwargs):
        raise RuntimeError('GPT-Eval은 이 Kaggle 평가에 포함되지 않습니다.')
GeminiEvaluator = GPTEvaluator
system_message = user_message = jobs = None
"""
assert old in s or shim in s
s=s.replace(old,shim)
s=s.replace('AutoTokenizer.from_pretrained(cfg.model_path)','AutoTokenizer.from_pretrained(cfg.model_path, use_fast=False)')
s=s.replace('attn_implementation="flash_attention_2"','attn_implementation="sdpa"')
s=s.replace('if "llava_phi" in cfg.model_family:','if "llava-phi" in cfg.model_family:')
old_load='model.load_state_dict(torch.load(cfg.LoRA.lora_path), strict=False)'
new_load="""load_result = model.load_state_dict(torch.load(cfg.LoRA.lora_path, map_location='cpu', weights_only=True), strict=False)
            if load_result.unexpected_keys:
                raise RuntimeError(f'Checkpoint mismatch: {load_result.unexpected_keys[:10]}')"""
assert old_load in s or new_load in s
s=s.replace(old_load,new_load)
old_cuda='    model.half().cuda()'
new_cuda='    model.half().cuda()\n    model.eval()'
assert old_cuda in s or new_cuda in s
s=s.replace(old_cuda,new_cuda)
# run_generation의 문자열 answer-tag 분리는 LLaVA-Phi 토크나이저의 공백/개행 복원에 취약합니다.
old_generation = '''    input_strings = tokenizer.batch_decode(input_ids)

    model_config = get_model_identifiers_from_yaml(cfg.model_family)
    question_start_tag = model_config['question_start_tag']
    answer_tag = model_config['answer_tag']
    answer_tag = answer_tag.replace(\"\\n\", \"\") 

    ground_truth = [s.split(answer_tag)[1].strip(\" \") for s in input_strings]
    input_strings = [s.split(answer_tag)[0].strip(\" \") for s in input_strings]
    input_strings = [s + answer_tag for s in input_strings]
'''
new_generation = '''    labels = batch[\"labels\"]
    input_strings = []
    ground_truth = []
    for ids, sample_labels in zip(input_ids, labels):
        supervised = (sample_labels != -100).nonzero(as_tuple=False).flatten()
        if supervised.numel() == 0:
            raise RuntimeError(\"평가 샘플에 정답 토큰이 없습니다.\")
        answer_start = int(supervised[0])
        input_strings.append(tokenizer.decode(ids[:answer_start], skip_special_tokens=False))
        ground_truth.append(tokenizer.decode(sample_labels[supervised], skip_special_tokens=True).strip())

    model_config = get_model_identifiers_from_yaml(cfg.model_family)
    question_start_tag = model_config['question_start_tag']
    answer_tag = model_config['answer_tag']
'''
assert old_generation in s or new_generation in s
s=s.replace(old_generation,new_generation)
ast.parse(s)
backup=path.with_suffix('.py.before_kaggle_eval')
if not backup.exists(): backup.write_text(path.read_text())
path.write_text(s)
print('평가 호환성 패치 완료; 로컬/GitHub Python 파일에는 영향 없음')

In [ ]:
# 별도 eval config를 작성합니다. 원본 config/eval.yaml은 유지합니다.
from pathlib import Path
import yaml
repo=Path('/kaggle/working/FIUBench_kaggle')
cfg=yaml.safe_load((repo/'config/eval.yaml').read_text())
cfg['model_path']='gray311/vlm_unlearning_ft_llava_phi_3_mini'
cfg['LoRA']['lora_path']='/kaggle/working/ga_forget5_checkpoint/checkpoint.pt'
cfg['data_path']=['./dataset/full.json']*3
cfg['split_list']=['forget5','forget5','retain5']
cfg['question_key']=['question']*3
cfg['robust_question_key']=['paraphrased_question','question','question']
cfg['answer_key']=['answer']*3
cfg['base_answer_key']=['paraphrased_answer']*3
cfg['perturbed_answer_key']=['perturbed_answer']*3
cfg['eval_task']=['eval_forget_ape','eval_forget_log','eval_retain_log']
cfg['robust_eval']=[['ape'],['exact_match','mink'],['rouge']]
cfg['workers']=0
cfg['batch_size']=1
cfg['perturb_batch_size']=1
cfg['overwrite']=False
cfg['save_dir']='/kaggle/working/ga_forget5_checkpoint/eval_results'
path=repo/'config/eval_ga_forget5_kaggle.yaml'
path.write_text(yaml.safe_dump(cfg,sort_keys=False,allow_unicode=True))
print('평가 config:',path)
print('tasks:',cfg['eval_task'])
print('checkpoint:',cfg['LoRA']['lora_path'])

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/FIUBench_kaggle
export HF_HOME=/kaggle/working/hf_cache
export TOKENIZERS_PARALLELISM=false
export PYTHONUNBUFFERED=1
export HYDRA_FULL_ERROR=1
python evaluate_util.py --config-name eval_ga_forget5_kaggle 2>&1 | tee /kaggle/working/ga_forget5_eval_console.log
printf '\nEVAL_PROCESS_EXIT_0\n'

In [ ]:
import json
from pathlib import Path
p=Path('/kaggle/working/ga_forget5_checkpoint/eval_results')
for name in ['forget5_eval_forget_ape.json','forget5_eval_forget_log.json','retain5_eval_retain_log.json','retain5_eval_log_aggregated.json']:
    f=p/name
    assert f.is_file(),f'평가 파일 누락: {f}'
    obj=json.loads(f.read_text())
    print(name, f.stat().st_size, 'bytes', list(obj)[:12])
print('핵심 FIUBench 평가 파일 생성 확인. GPT-Eval/MME/POPE/KS는 별도 실행 필요.')

평가 종료 후 `/kaggle/working/ga_forget5_checkpoint/eval_results`와 `/kaggle/working/ga_forget5_eval_console.log`를 저장하세요. 출력에는 `generated_text`가 들어 있어 평가 데이터가 포함될 수 있습니다. 원본 평가 코드는 `cfg.save_dir`을 checkpoint 폴더 밑으로 재설정하므로, 이 노트북은 checkpoint를 `/kaggle/working`으로 복사했습니다.